# Self-Instruct Replication: Qwen3.5:4b

Replikasi pipeline Self-Instruct (Wang et al., 2023) menggunakan Qwen3.5:4b lokal via Ollama,
dengan LoRA fine-tuning di atas model base, dan evaluasi dibandingkan hasil paper asli.

## Overview Pipeline

Notebook ini terdiri dari tiga tahap utama:

**Tahap 1: Data Generation** (notebook ini)

Pipeline Self-Instruct empat langkah dijalankan menggunakan `qwen3.5:4b` via Ollama sebagai
model generator:
1. Instruction Generation: bootstrap instruksi baru dari seed pool
2. Classification Task Identification: tentukan tipe output (klasifikasi atau bebas)
3. Instance Generation: buat pasangan input-output untuk tiap instruksi
4. Filtering & Postprocessing: saring data, deduplikasi, format ke `{prompt, completion}`

Output: `finetuning_data.jsonl`

**Tahap 2: LoRA Fine-Tuning**

Model target: `unsloth/Qwen3.5-4B-Base`. Fine-tuning dengan LoRA via Unsloth, dijalankan di
**vast.ai** pada instance GPU **RTX 4000-series**. Pemilihan 4000-series dikarenakan
stack CUDA/torch untuk seri diatas ini belum didukung penuh oleh Unsloth.

**Tahap 3: Evaluasi**

Hasil fine-tuning dievaluasi pada:
- Subset **SuperNI** (ROUGE-L, zero-shot), metrik yang sama dengan paper §4.3
- **User-oriented tasks**, rubrik A/B/C/D, human evaluation bertiga, sesuai paper §4.4

Target perbandingan dari paper asli (Table 3):

| Model | ROUGE-L |
|---|---|
| GPT-3 vanilla (baseline) | 6.8 |
| GPT-3 + SELF-INSTRUCT | 39.9 |
| InstructGPT001 | 40.8 |

## Perbedaan dari Paper Asli

| Aspek | Paper (Wang et al., 2023) | Replikasi ini |
|---|---|---|
| Generator | GPT-3 davinci (175B) | Qwen3.5:4b (4B) via Ollama |
| Fine-tune target | GPT-3 davinci (sama) | Qwen3.5-4B-Base (sama) |
| Seed selection | 175 task tulis manual | 175 task via K-Means clustering dari Dolly-15k |
| Filtering | Heuristik + ROUGE-L | Heuristik + ROUGE-L + embedding similarity |
| Fine-tuning method | Full fine-tuning (OpenAI API) | LoRA (Unsloth) |
| Skala data | ~52K instruksi | Dikurangi sesuai resource |

**Pertanyaan replikasi**: Apakah pipeline Self-Instruct tetap menghasilkan peningkatan
instruction-following yang signifikan ketika model generator jauh lebih kecil (4B vs 175B)?

## Anggota Kelompok:

- **Kevin Pratama**: Seed preparation, Step 1, Step 2
- **Alfi Maulana Akbar**: Ollama setup, Step 3, pipeline orchestration
- **Maulana Yusuf Habibi**: Step 4, LoRA fine-tuning, evaluasi

## Setup

Cell `%pip install` di bawah memasang dependency Python untuk Tahap 1 (client Ollama, sentence-transformers, scikit-learn, rouge-score, datasets, numpy).

Ollama (server + model) tidak perlu disiapkan manual saat jalan di vast.ai. Cell orkestrasi pipeline (Pipeline Utama) otomatis meng-install Ollama, menjalankan `ollama serve`, dan men-download `qwen3.5:4b` jika belum ada.

> **Catatan Qwen3.5**: Qwen3.5 punya mode "extended thinking" yang aktif secara default. Pipeline ini menonaktifkan thinking via opsi `think: false` di setiap request agar output bersih dan tidak menyertakan blok `<think>...</think>` yang akan merusak parsing.

In [1]:
%pip install ollama sentence-transformers scikit-learn rouge-score datasets numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 22.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 86.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 61.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 100.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 90.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 78.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 104.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.3 MB/s  0:00:00
   ━━━━

## Import

- `ollama`: client buat panggil Qwen lokal
- `sentence_transformers`: embedding (dipakai di seed selection dan filtering)
- `sklearn.cluster.KMeans`: clustering buat pilih seed
- `rouge_score`: ROUGE-L untuk filter similarity (sesuai paper)
- `datasets`: load Dolly-15k sebagai candidate pool seed

In [2]:
import json
import random
import numpy as np
import ollama
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from rouge_score import rouge_scorer
from datasets import load_dataset

random.seed(42)
np.random.seed(42)

## Helper: Panggil Qwen via Ollama

Wrapper yang memanggil Qwen pada setiap step dengan temperature, max tokens, dan stop sequences yang sesuai.

In [3]:
MODEL = 'qwen3.5:4b'


def call_qwen(prompt, temperature=0.0, max_tokens=512, stop=None):
    options = {
        'temperature': temperature,
        'num_predict': max_tokens,
    }
    if stop:
        options['stop'] = stop
    response = ollama.chat(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        think=False,
        options=options
    )
    return response['message']['content']

## Seed Selection: Embedding + Clustering

Paper asli pakai 175 seed yang ditulis manual oleh penulis dan labmates di UW. Kelemahannya yaitu subjektif dan susah direplikasi.

Untuk pendekatan yang kami gunakan:
1. Ambil 2000 instruksi acak dari Dolly-15k sebagai candidate pool
2. Embed semua instruksi pakai sentence transformer
3. K-Means clustering jadi 175 cluster
4. Ambil instruksi yang paling dekat ke centroid tiap cluster

Hasilnya: 175 seed yang secara semantik beragam, dipilih dengan kriteria objektif.

In [4]:
def load_candidate_seeds(n=2000):
    """Ambil n instruksi acak dari Dolly-15k sebagai candidate pool."""
    ds = load_dataset('databricks/databricks-dolly-15k', split='train')
    samples = ds.shuffle(seed=42).select(range(min(n, len(ds))))
    return [{'instruction': s['instruction']} for s in samples]


def select_seeds_by_clustering(candidates, n_seeds, embedder):
    """Pilih n_seeds instruksi paling beragam via K-Means clustering."""
    instructions = [c['instruction'] for c in candidates]
    embeddings = embedder.encode(instructions, show_progress_bar=True)

    kmeans = KMeans(n_clusters=n_seeds, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)

    selected = []
    for cluster_id in range(n_seeds):
        cluster_idx = np.where(labels == cluster_id)[0]
        cluster_embs = embeddings[cluster_idx]
        centroid = kmeans.cluster_centers_[cluster_id]
        distances = np.linalg.norm(cluster_embs - centroid, axis=1)
        closest = cluster_idx[np.argmin(distances)]
        selected.append(candidates[closest])
    return selected


embedder = SentenceTransformer('all-MiniLM-L6-v2')
candidates = load_candidate_seeds(n=2000)
seeds = select_seeds_by_clustering(candidates, n_seeds=175, embedder=embedder)

print(f'Selected {len(seeds)} seeds')
print('Contoh 3 seed pertama:')
for s in seeds[:3]:
    print(f'  - {s["instruction"]}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Selected 175 seeds
Contoh 3 seed pertama:
  - Create a dialogue between two characters discussing the impact of social media on their lives. Your dialogue should explore how to compare social media to real life.
  - Give me a list of 1980s computer games.
  - Why do people like to travel?


## Step 1: Instruction Generation

Sampel 8 instruksi dari task pool (6 dari seed manusia, 2 dari instruksi yang sudah dihasilkan model), masukkan ke prompt, minta model bikin lanjutannya. Stop sequence di "Task 16". Pada awal pipeline belum ada machine task, jadi semua 8 contoh diambil dari seed.

`INSTRUCTION_PROMPT` juga secara eksplisit mendorong variasi tipe tugas (klasifikasi, rewriting, ekstraksi, summarization, brainstorming, generasi bebas) supaya hasil tidak menyempit ke pertanyaan-jawaban faktual saja. `parse_instructions()` mengambil tiap baris berformat `Task N: ...` dari output model dan membuang penanda `Task N:`-nya.

In [5]:
import re

INSTRUCTION_PROMPT = """Continue the task list below. Output only task descriptions in the exact same format, one per line. Do not add introductions, explanations, or summaries. Include a variety of task types (classification, rewriting, extraction, summarization, brainstorming, open-ended generation), not only factual question-answering.

{examples}
Task {next_num}:"""


def parse_instructions(response):
    instructions = []
    for line in response.split('\n'):
        line = line.strip().lstrip('*').strip()
        if not line.startswith('Task'):
            continue
        parts = line.split(':', 1)
        if len(parts) == 2 and parts[1].strip():
            instructions.append(parts[1].strip().strip('*').strip())
    return instructions


def generate_instructions(task_pool):
    human_tasks = [t for t in task_pool if t['source'] == 'human']
    machine_tasks = [t for t in task_pool if t['source'] == 'machine']

    sampled_human = random.sample(human_tasks, 6)
    if len(machine_tasks) >= 2:
        sampled_machine = random.sample(machine_tasks, 2)
    else:
        sampled_machine = random.sample(human_tasks, 2)

    examples = sampled_human + sampled_machine
    random.shuffle(examples)

    example_text = '\n'.join(
        f'Task {i+1}: {ex["instruction"]}'
        for i, ex in enumerate(examples)
    )
    prompt = INSTRUCTION_PROMPT.format(
        examples=example_text,
        next_num=len(examples) + 1
    )

    response = call_qwen(
        prompt,
        temperature=0.7,
        max_tokens=1024,
        stop=['Task 16:']
    )
    return parse_instructions(response)

## Step 2: Classification Task Identification

Cek tiap instruksi: apakah ini task klasifikasi (output terbatas) atau bebas? Pakai few-shot prompt dengan 6 contoh. Hal ini dilakukan karena Step 3 punya dua jalur berbeda untuk klasifikasi vs non-klasifikasi.

In [6]:
CLASSIFICATION_PROMPT = """Can the following task be regarded as a classification task with finite output labels?

Task: Tell me if the statement is true or false.
Is it classification? Yes

Task: Give me an example of a time when you had to use your sense of humor.
Is it classification? No

Task: Detect if the Reddit thread contains hate speech.
Is it classification? Yes

Task: Rank these countries by their population.
Is it classification? No

Task: Does the document support the claim? Answer with Support or Unsupport.
Is it classification? Yes

Task: Create a detailed budget for the given hypothetical trip.
Is it classification? No

Task: {instruction}
Is it classification?"""


def is_classification(instruction):
    prompt = CLASSIFICATION_PROMPT.format(instruction=instruction)
    response = call_qwen(prompt, temperature=0.0, max_tokens=5, stop=['\n', 'Task:'])
    return response.strip().lower().startswith('yes')

## Step 3: Instance Generation

Untuk tiap instruksi yang lolos filter, generate satu pasangan input-output:

- **Non-klasifikasi**: input-first (generate input dulu, baru output)
- **Klasifikasi**: output-first (generate label dulu, baru input). Dilakukan agar distribusi label seimbang dan tidak bias ke satu kelas.

In [7]:
INPUT_FIRST_PROMPT = """For the following task, generate one example with input and output. If no input is needed, write "Input: None".

Task: {instruction}

Format your response exactly as:
Input: <the input or None>
Output: <the output>
"""


OUTPUT_FIRST_PROMPT = """For the following classification task, generate one example by first writing a class label, then writing an input that fits that label.

Task: {instruction}

Format your response exactly as:
Output: <the class label>
Input: <the input>
"""


def parse_instance(text):
    lines = text.strip().split('\n')
    result = {'input': '', 'output': ''}
    current = None
    for line in lines:
        if line.startswith('Input:'):
            current = 'input'
            result['input'] = line[len('Input:'):].strip()
        elif line.startswith('Output:'):
            current = 'output'
            result['output'] = line[len('Output:'):].strip()
        elif current and line.strip():
            result[current] += '\n' + line
    if result['input'].lower() == 'none':
        result['input'] = ''
    return result


def generate_instance(instruction, is_clf):
    template = OUTPUT_FIRST_PROMPT if is_clf else INPUT_FIRST_PROMPT
    prompt = template.format(instruction=instruction)
    response = call_qwen(prompt, temperature=0.0, max_tokens=900)
    return parse_instance(response)

## Filter Helper: Instruction Filtering

Tiga lapis filter otomatis yang dijalankan tiap iterasi untuk menjaga kualitas instruksi yang masuk ke task pool:

1. **Heuristik dasar**: hapus yang terlalu pendek/panjang, mengandung keyword yang tidak bisa diproses model teks (image, picture, dll), atau merupakan teks template yang bocor (model kadang meng-echo prompt; ditolak lewat `TEMPLATE_LEAK_MARKERS`) sesuai paper §3.3
2. **ROUGE-L < 0.7** dibanding instruksi yang sudah ada sesuai paper §3.3 (threshold yang sama)
3. **Embedding similarity < 0.85** tambahan baru, lebih semantik dibanding n-gram matching

Filter ini sesuai paper §3.3 dan dieksekusi inline di pipeline utama. Sampling manual review (20 instruksi per iterasi) juga diambil di sini untuk annotasi.

In [8]:
BLACKLIST = ['image', 'picture', 'graph', 'flowchart', 'diagram', 'photo', 'video', 'audio']

# Penanda teks prompt/template yang bocor jadi "instruksi" (model echo prompt)
TEMPLATE_LEAK_MARKERS = [
    'continue the task list',
    'output only task',
    'one per line',
    'do not add introductions',
    'exact same format',
    'task descriptions',
]

SCORER = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)


def passes_heuristics(instruction):
    n_words = len(instruction.split())
    if n_words < 3 or n_words > 150:
        return False
    low = instruction.lower()
    if any(kw in low for kw in BLACKLIST):
        return False
    if any(m in low for m in TEMPLATE_LEAK_MARKERS):
        return False
    return True


def passes_rouge(new_inst, existing_insts, threshold=0.7):
    for existing in existing_insts:
        score = SCORER.score(existing, new_inst)['rougeL'].fmeasure
        if score >= threshold:
            return False
    return True


def passes_embedding(new_inst, existing_embs, embedder, threshold=0.85):
    new_emb = embedder.encode([new_inst])[0]
    sims = existing_embs @ new_emb / (
        np.linalg.norm(existing_embs, axis=1) * np.linalg.norm(new_emb) + 1e-8
    )
    return float(np.max(sims)) < threshold


def sample_for_manual_review(instructions, n=20):
    if len(instructions) <= n:
        return list(instructions)
    return random.sample(instructions, n)

## Orkestrasi: Pipeline Utama

Loop utama yang menyatukan semua step. Pada tiap iterasi dilakukan:

1. Generate batch instruksi baru (Step 1)
2. Filter otomatis (Step 4 partial: heuristik + ROUGE + embedding)
3. Untuk yang lolos: cek tipe (Step 2), generate instance (Step 3)
4. Sampel buat review manual disimpan terpisah
5. Save progress ke jsonl tiap iterasi

In [10]:
import subprocess, time

# Pin Ollama ke build pra-0.30 (CUDA 12, cocok dengan driver 535).
# 0.30.x bundel CUDA 13 yang tak bisa di-load driver 535 -> "device kernel image is invalid".
subprocess.run(["bash", "-c",
    "curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.24.0 sh"], check=True)

# Restart bersih: matikan server lama (mungkin versi 0.30.x stale) lalu start ulang
subprocess.run(["pkill", "-x", "ollama"], capture_output=True)
time.sleep(2)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if "qwen3.5:4b" not in result.stdout:
    subprocess.run(["ollama", "pull", "qwen3.5:4b"], check=True)

print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout)
print("Ollama siap.")

def run_pipeline(seeds, embedder, n_iterations=2,
                 output_path='generated_data.jsonl',
                 review_path='manual_review_sample.jsonl'):
    task_pool = [{'instruction': s['instruction'], 'source': 'human'} for s in seeds]
    existing_embs = embedder.encode(
        [t['instruction'] for t in task_pool], show_progress_bar=False
    )

    all_data = []
    review_samples = []

    for it in range(n_iterations):
        print(f'\n=== Iterasi {it+1}/{n_iterations} ===')

        new_raw = generate_instructions(task_pool)
        print(f'Generated mentah: {len(new_raw)}')

        existing_insts = [t['instruction'] for t in task_pool]
        kept = []
        for inst in new_raw:
            if not passes_heuristics(inst):
                continue
            if not passes_rouge(inst, existing_insts):
                continue
            if not passes_embedding(inst, existing_embs, embedder):
                continue
            kept.append(inst)
            existing_insts.append(inst)
            new_emb = embedder.encode([inst], show_progress_bar=False)
            existing_embs = np.vstack([existing_embs, new_emb])
        print(f'Lolos filter: {len(kept)}')

        for inst in kept:
            is_clf = is_classification(inst)
            instance = generate_instance(inst, is_clf)
            all_data.append({
                'instruction': inst,
                'is_classification': is_clf,
                'input': instance['input'],
                'output': instance['output'],
                'iteration': it + 1,
            })
            task_pool.append({'instruction': inst, 'source': 'machine'})

        sampled = sample_for_manual_review(kept)
        review_samples.extend(
            [{'iteration': it + 1, 'instruction': s} for s in sampled]
        )

        with open(output_path, 'w') as f:
            for item in all_data:
                f.write(json.dumps(item) + '\n')
        with open(review_path, 'w') as f:
            for item in review_samples:
                f.write(json.dumps(item) + '\n')

        print(f'Total task terkumpul: {len(all_data)}')

    return all_data

sample_seeds = random.sample(seeds, 8)
example_text = '\n'.join(f'Task {i+1}: {s["instruction"]}' for i, s in enumerate(sample_seeds))
prompt = INSTRUCTION_PROMPT.format(examples=example_text, next_num=9)

response = call_qwen(prompt, temperature=0.7, max_tokens=1024, stop=['Task 16:'])
print(repr(response))
print(parse_instructions(response))

# Yield ~1.5 instruksi/iterasi -> 500 iterasi kira-kira ~750 instruksi.
# Pipeline menyimpan progres tiap iterasi (aman bila terputus). Untuk smoke test
# turunkan dulu ke n_iterations=2 sebelum jalan penuh.
data = run_pipeline(seeds, embedder, n_iterations=500)
print(f'\nSelesai. Total: {len(data)} task')

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


ollama version is 0.24.0

Ollama siap.
'Task 9: Rewrite the sentence "The cat sat on the mat" using a simile.\nTask 10: Extract the main subject and predicate from the sentence: "The quick brown fox jumps over the lazy dog."\nTask 11: Summarize the following paragraph in one sentence: "Climate change is causing rising global temperatures, leading to more frequent extreme weather events, which in turn threatens ecosystems and human communities worldwide."\nTask 12: Brainstorm five potential causes for the recent increase in remote work adoption.\nTask 13: Generate a list of adjectives that describe the color blue.\nTask 14: Translate the phrase "Miles apart" into French.\nTask 15: Categorize the following items into three groups: fruit, vegetables, and grains: apple, carrot, rice, banana, potato, wheat, orange, corn.'
['Rewrite the sentence "The cat sat on the mat" using a simile.', 'Extract the main subject and predicate from the sentence: "The quick brown fox jumps over the lazy dog."

## Step 4: Prepare for Finetuning

Sesuai paper asli (Wang et al., 2023 §3.4), Step 4 mengonversi data yang terkumpul menjadi format `{prompt, completion}` siap fine-tuning:

1. **Bersihkan input redundan**: kalau `input` mengulang kalimat instruksi (kemiripan `difflib` > 0.6), input dianggap kosong supaya tidak mengajari model pola yang membingungkan
2. **Filter instance tidak valid**: output kosong, input == output, output terlalu pendek, terlalu panjang, atau model menolak (dimulai dengan "I cannot", "I'm sorry")
3. **Deduplication**: buang pasangan (instruction, input) yang identik
4. **Random template**: format ulang pakai template acak (varian dengan/tanpa input) supaya model lebih robust terhadap variasi format prompt

In [11]:
import difflib

# Template prompt dengan input (dipakai kalau instruksi butuh konteks)
PROMPT_TEMPLATES_WITH_INPUT = [
    "{instruction}\n\nInput:\n{input}\n\nOutput:\n",
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n",
    "Task: {instruction}\nInput: {input}\nOutput: ",
]

# Template prompt tanpa input (instruksi berdiri sendiri)
PROMPT_TEMPLATES_NO_INPUT = [
    "{instruction}\n\nOutput:\n",
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n",
    "Task: {instruction}\nOutput: ",
]


def prepare_for_finetuning(data, output_path='finetuning_data.jsonl'):
    """Konversi data terkumpul ke format {prompt, completion} untuk fine-tuning (paper Step 4)."""
    results = []
    seen = set()

    for item in data:
        inst = item['instruction'].strip()
        inp = item.get('input', '').strip()
        out = item.get('output', '').strip()
        out = out.replace('\\n', '\n')

        # Kalau input cuma mengulang instruksi, anggap tidak ada input
        if inp and difflib.SequenceMatcher(None, inp.lower(), inst.lower()).ratio() > 0.6:
            inp = ''

        context_required = ['the provided', 'the following text', 'the article', 'the passage', 'the excerpt']
        if any(phrase in inst.lower() for phrase in context_required) and not inp:
            continue

        # Filter instance tidak valid
        if not out:
            continue
        if len(out.split()) < 2:
            continue
        if len(inst.split()) + len(inp.split()) + len(out.split()) > 2000:
            continue
        if inp and inp.lower() == out.lower():
            continue
        low_out = out.lower()
        if low_out.startswith("i'm sorry") or low_out.startswith("i cannot") or low_out.startswith("i'm not able"):
            continue

        # Deduplication berdasarkan (instruction, input)
        key = (inst.lower(), inp.lower(), out[:100].lower())
        if key in seen:
            continue
        seen.add(key)

        # Format dengan random template
        if inp:
            template = random.choice(PROMPT_TEMPLATES_WITH_INPUT)
            prompt = template.format(instruction=inst, input=inp)
        else:
            template = random.choice(PROMPT_TEMPLATES_NO_INPUT)
            prompt = template.format(instruction=inst)

        results.append({'prompt': prompt, 'completion': ' ' + out})

    # Simpan ke file
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in results:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    return results


finetuning_data = prepare_for_finetuning(data)
print(f'Total instances setelah Step 4: {len(finetuning_data)}')
print(f'Drop rate: {1 - len(finetuning_data)/max(len(data),1):.1%}')
print('\nContoh instance siap fine-tuning:')
if finetuning_data:
    s = finetuning_data[0]
    print(f"Prompt     : {repr(s['prompt'][:200])}")
    print(f"Completion : {repr(s['completion'][:100])}")
print('\nDisimpan ke finetuning_data.jsonl')

import os
print('Saved to', os.path.abspath('finetuning_data.jsonl'))

Total instances setelah Step 4: 970
Drop rate: 25.8%

Contoh instance siap fine-tuning:
Prompt     : 'Rewrite the sentence "The cat sat on the mat" to sound more poetic.\n\nOutput:\n'
Completion : ' Upon the woven mat, the cat reclined in quiet repose.'

Disimpan ke finetuning_data.jsonl
Saved to /workspace/finetuning_data.jsonl


## Fine-tuning dengan Unsloth

Fine-tune `unsloth/Qwen3.5-4B-Base` di atas data self-instruct yang dihasilkan pipeline di atas. Menggunakan:
- **Unsloth** + **LoRA** untuk training yang efisien
- **`FastLanguageModel`** bukan `FastVisionModel` karena data kita teks saja (bukan gambar)
- **SFTTrainer** dari TRL dengan format `{prompt, completion}` digabung ke field `text`

### Lingkungan: vast.ai, GPU RTX 4000-series

Bagian fine-tuning ini dijalankan di **vast.ai**. **Diutamakan instance dengan GPU RTX 4000-series**, karena adanya keperluan kompatibilitas CUDA. GPU yang lebih baru (5000-series/Blackwell) butuh CUDA versi paling baru (cu13x) yang otomatis menarik `torch >= 2.12`, dan versi torch itu **memecah stack kompilasi Unsloth**. RTX 4000-series adalah generasi terbaru yang masih didukung penuh oleh torch versi kompatibel, sehingga training jalan tanpa harus menambal versi.

Dua cell instalasi di bawah:
1. Install `unsloth`, `unsloth_zoo`, dan `trl` lewat `uv` (resolver yang cepat).
2. Force-reinstall `torch`/`torchvision`/`torchaudio` agar cocok dengan CUDA instance vast.ai dan kompatibel dengan Unsloth.

In [2]:
%pip install --upgrade -qqq uv
%pip install -U "datasets==4.3.0" dill
import sys
!uv pip install --python {sys.executable} -qqq unsloth unsloth_zoo
!uv pip install --python {sys.executable} --upgrade --no-deps "trl>=0.18.2,<=0.24.0" unsloth unsloth_zoo

Note: you may need to restart the kernel to use updated packages.
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.13 environment at: /venv/main
Resolved 3 packages in 135ms                                         
Checked 3 packages in 0.20ms


In [4]:
import sys
# torch + ekosistemnya sekaligus, semua build cu128 dari index PyTorch.
# xformers digabung di sini supaya terkunci ke torch 2.10 (bukan ketinggalan di 0.0.29.post3).
!{sys.executable} -m pip install --force-reinstall --no-cache-dir \
    --index-url https://download.pytorch.org/whl/cu128 \
    "torch==2.10.0" "torchvision==0.25.0" "torchaudio==2.10.0" "xformers==0.0.35"

# fsspec turunkan agar cocok datasets 4.3.0 (dari PyPI, terpisah karena tidak ada di index PyTorch)
!{sys.executable} -m pip install --no-cache-dir "fsspec[http]<=2025.9.0"

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 95.3 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 120.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 99.4 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 93.1 MB/s  0:00:06:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 95.9 MB/s  0:00:02ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 90.0 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 88.1 MB/s  0:00:03ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 92.6 MB/s  0:00:03ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 93.1 MB/s  0:00:03:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 M

### Load Model dan LoRA Adapters

**Setelah dua cell instalasi di atas selesai, restart sesi/kernel dulu** sebelum menjalankan cell di bawah (supaya torch/unsloth versi baru benar-benar terpakai).

Dua cell di bawah:
1. `HfApi.list_models(...)` digunakan untuk mengecek repo Unsloth Qwen3.5 yang tersedia. Opsional, boleh dilewati.
2. Load model + pasang LoRA (cell utama).

Detail cell load model:
- `UNSLOTH_COMPILE_DISABLE=1` + hapus `unsloth_compiled_cache` mematikan kompilasi yang sering bentrok dengan kombinasi torch/CUDA tertentu di vast.ai (ini bagian dari alasan mengutamakan GPU 4000-series).
- `FastLanguageModel` (bukan `FastVisionModel`) karena data self-instruct hanya teks, tidak ada gambar.
- `load_in_4bit=True` mengkuantisasi model ke 4bit on-the-fly (tidak ada repo pre-quantized untuk `Qwen3.5-4B-Base`), menghemat VRAM.
- LoRA hanya melatih ~1% parameter: rank 16, alpha 16, dropout 0.05, target semua proyeksi attention + MLP.

In [6]:
from huggingface_hub import HfApi
api = HfApi()
for m in api.list_models(author="unsloth", search="Qwen3.5"):
    print(m.id)

unsloth/Qwen3.5-9B-GGUF
unsloth/Qwen3.5-9B-MTP-GGUF
unsloth/Qwen3.5-4B-MTP-GGUF
unsloth/Qwen3.5-122B-A10B-GGUF
unsloth/Qwen3.5-0.8B
unsloth/Qwen3.5-0.8B-MTP-GGUF
unsloth/Qwen3.5-397B-A17B-GGUF
unsloth/Qwen3.5-2B-GGUF
unsloth/Qwen3.5-4B-GGUF
unsloth/Qwen3.5-2B-MTP-GGUF
unsloth/Qwen3.5-35B-A3B-GGUF
unsloth/Qwen3.5-122B-A10B
unsloth/Qwen3.5-0.8B-GGUF
unsloth/Qwen3.5-4B-Base
unsloth/Qwen3.5-35B-A3B-MTP-GGUF
unsloth/Qwen3.5-122B-A10B-MTP-GGUF
unsloth/Qwen3.5-397B-A17B
unsloth/Qwen3.5-27B-GGUF
unsloth/Qwen3.5-35B-A3B
unsloth/Qwen3.5-27B
unsloth/Qwen3.5-35B-A3B-Base
unsloth/Qwen3.5-397B-A17B-FP8
unsloth/Qwen3.5-35B-A3B-Experiments-GGUF
unsloth/Qwen3.5-9B
unsloth/Qwen3.5-2B
unsloth/Qwen3.5-4B
unsloth/Qwen3.5-0.8B-Base
unsloth/Qwen3.5-2B-Base
unsloth/Qwen3.5-9B-Base
unsloth/Qwen3.5-27B-MTP-GGUF
unsloth/Qwen3.5-397B-A17B-MTP-GGUF


In [7]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

import shutil
shutil.rmtree("/workspace/unsloth_compiled_cache", ignore_errors=True)

from unsloth import FastLanguageModel
import torch

print(torch.__version__)

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3.5-4B-Base",
    max_seq_length=2048,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,  # dari 0 -> 0.05 untuk meredam overfitting pada data kecil
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
2.10.0+cu128
==((====))==  Unsloth 2026.6.1: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.648 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/146 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


### Siapkan Dataset

Load `finetuning_data.jsonl` hasil Step 4. SFTTrainer butuh satu field `text` berisi prompt + completion yang sudah digabung. Dua hal penting di sini:

- **EOS eksplisit**: tiap `text` ditutup dengan token EOS supaya model pasti belajar berhenti, bukan kebetulan.
- **Split train/validation 90/10**: validation set dipakai untuk mengukur overfitting (lihat `eval_loss` per epoch saat training, dan evaluasi held-out base vs fine-tuned setelahnya).

In [8]:
import json
from datasets import Dataset

with open('finetuning_data.jsonl', 'r', encoding='utf-8') as f:
    raw = [json.loads(line) for line in f]

# Tambahkan EOS eksplisit supaya model pasti belajar berhenti
EOS = getattr(tokenizer, "eos_token", None) or "<|endoftext|>"
records = [{'text': item['prompt'] + item['completion'] + EOS} for item in raw]
hf_dataset = Dataset.from_list(records)

# Split train/validation 90/10 supaya overfitting terukur
split = hf_dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = split['train']
eval_dataset = split['test']

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print('\nContoh text field pertama (300 karakter):')
print(train_dataset[0]['text'][:300])

Train: 873 | Eval: 97

Contoh text field pertama (300 karakter):
Generate five unique business names for a coffee shop located in a mountainous region.

Output:
 1. Summit Sips
2. Peak Brew Collective
3. Alpine Aroma Roasters
4. Cloud Nine Coffee
5. Highland Hearth<|endoftext|>


### Training

Training `num_train_epochs=2` di atas split train, dengan evaluasi tiap epoch (`eval_strategy="epoch"`) di atas split validation. Diturunkan dari 3 ke 2 epoch karena run v2 menunjukkan overfitting: `eval_loss` mulai naik setelah epoch 1 (0.74 -> 0.75 -> 0.84) sementara `train_loss` terus turun.

Sebagai pengaman, `load_best_model_at_end=True` dengan `metric_for_best_model="eval_loss"` membuat model yang dipakai untuk evaluasi dan disimpan adalah checkpoint dengan `eval_loss` terbaik, bukan otomatis epoch terakhir. `save_strategy` disetel `"epoch"` agar cocok dengan `eval_strategy` (syarat `load_best_model_at_end`).

In [9]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,          # run v2 overfit: eval_loss naik setelah epoch 1, turunkan 3 -> 2
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        max_seq_length=2048,
        dataset_text_field="text",
        eval_strategy="epoch",       # evaluasi tiap epoch -> eval_loss terukur
        save_strategy="epoch",       # wajib sama dengan eval_strategy agar load_best_model_at_end jalan
        load_best_model_at_end=True, # pakai checkpoint dengan eval_loss terbaik, bukan epoch terakhir
        metric_for_best_model="eval_loss",
        greater_is_better=False,     # eval_loss makin kecil makin baik
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

import warnings
warnings.filterwarnings("ignore", message=".*_check_is_size.*")

import torch._dynamo
torch._dynamo.config.disable = True

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

Unsloth: Tokenizing ["text"] (num_proc=28):   0%|          | 0/873 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=28):   0%|          | 0/97 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248044}.


GPU = NVIDIA GeForce RTX 4090. Max memory = 23.648 GB.
8.568 GB of memory reserved.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 873 | Num Epochs = 2 | Total steps = 220
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.354754,0.554378
2,0.175632,0.558377


Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-110/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-220/tokenizer_config.json.


640.2572 seconds used for training.
10.67 minutes used for training.
Peak reserved memory = 11.674 GB.
Peak reserved memory for training = 3.106 GB.
Peak reserved memory % of max memory = 49.366 %.
Peak reserved memory for training % of max memory = 13.134 %.


### Evaluasi: Held-out + Baseline

Pada bagian ini, daripada menguji ulang contoh data latih (yang cuma mengukur hafalan), kelompok kami melakukan evaluasi pada **validation set** (instruksi yang tidak ikut dilatih) dan membandingkan **model base vs hasil fine-tune**.

Caranya: `model.disable_adapter()` mematikan LoRA sementara, jadi bisa dibandingkan dua kondisi tanpa memuat dua model. Generasi memakai **greedy** (`do_sample=False`) supaya deterministik dan bisa diulang.

Interpretasi hasil: jika ROUGE-L fine-tuned lebih tinggi dari base, Self-Instruct memberi peningkatan instruction-following. Kalau tidak, itu juga temuan valid (model 4B mungkin terlalu kecil).

> **Catatan**: validation set ini berasal dari distribusi data generasi yang sama (in-distribution held-out). Untuk klaim sesuai paper, lanjutkan ke **SuperNI subset** (ROUGE-L) di Tahap 3. Mekanisme `generate_text` + `disable_adapter()` di bawah bisa dipakai ulang, tinggal ganti sumber prompt.

In [10]:
import contextlib
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)


def generate_text(prompt, use_adapter=True, max_new_tokens=128):
    """Generate deterministik (greedy). use_adapter=False = model base tanpa LoRA."""
    inputs = text_tok(prompt, return_tensors="pt").to("cuda")
    ctx = contextlib.nullcontext() if use_adapter else model.disable_adapter()
    with ctx:
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # greedy, supaya evaluasi bisa diulang
            use_cache=True,
        )
    gen = out[0][inputs['input_ids'].shape[1]:]
    return text_tok.decode(gen, skip_special_tokens=True).strip()


# Penanda akhir-prompt untuk SEMUA template (urut dari yang paling spesifik).
# Versi v2 hanya mengenali 2 dari 6 template, jadi 26/84 sampel terlewat (cuma 58 terhitung).
RESPONSE_MARKERS = ["### Response:\n", "\n\nOutput:\n", "\nOutput: ", "\nOutput:\n"]


def split_prompt_ref(text):
    """Pisahkan prompt dan jawaban referensi. None kalau tidak ada penanda yang cocok."""
    for marker in RESPONSE_MARKERS:
        if marker in text:
            prompt, ref = text.split(marker, 1)
            return prompt + marker, ref
    return None, None


# Evaluasi pada validation set (instruksi yang TIDAK dilatih)
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_base, rouge_ft = [], []
skipped = 0

for ex in tqdm(eval_dataset, desc="Eval base vs ft"):
    prompt, ref = split_prompt_ref(ex['text'])
    if prompt is None:
        skipped += 1
        continue
    ref = ref.replace(EOS, '').strip()

    out_base = generate_text(prompt, use_adapter=False)
    out_ft = generate_text(prompt, use_adapter=True)
    rouge_base.append(scorer.score(ref, out_base)['rougeL'].fmeasure)
    rouge_ft.append(scorer.score(ref, out_ft)['rougeL'].fmeasure)

import numpy as np
print(f'Jumlah sampel eval: {len(rouge_ft)} (terlewat: {skipped} dari {len(eval_dataset)})')
print(f'ROUGE-L base (tanpa fine-tune) : {np.mean(rouge_base):.4f}')
print(f'ROUGE-L fine-tuned            : {np.mean(rouge_ft):.4f}')
print(f'Selisih (fine-tuned - base)   : {np.mean(rouge_ft) - np.mean(rouge_base):+.4f}')

Eval base vs ft:   0%|          | 0/97 [00:00<?, ?it/s]

Jumlah sampel eval: 97 (terlewat: 0 dari 97)
ROUGE-L base (tanpa fine-tune) : 0.1863
ROUGE-L fine-tuned            : 0.5370
Selisih (fine-tuned - base)   : +0.3507


### Evaluasi: SuperNI subset (ROUGE-L, zero-shot) dari paper §4.3

Evaluasi sesuai metrik utama paper: test split Super-NaturalInstructions (119 held-out task),
zero-shot (hanya `definition` + `inputs`, tanpa demonstrasi). Bandingkan base vs fine-tuned
menggunakan mekanisme `generate_text` + `disable_adapter()` yang sama. Target perbandingan paper
(Table 3): GPT-3 vanilla 6.8 -> +SELF-INSTRUCT 39.9.

In [11]:
from datasets import load_dataset
from collections import defaultdict
from tqdm.auto import tqdm
import numpy as np
import random

# Subset agar tractable. Paper pakai 119 task x sampai 100 instance. Naikkan untuk hasil lebih dekat paper.
N_TASKS, PER_TASK, MAX_SCAN = 30, 10, 20000

# Streaming + shuffle buffer: hindari muat 982k baris penuh, dan hindari bias urutan per-task
stream = load_dataset("Muennighoff/natural-instructions", split="test", streaming=True)
stream = stream.shuffle(seed=3407, buffer_size=10000)

buckets = defaultdict(list)
for n_scanned, row in enumerate(stream):
    if n_scanned >= MAX_SCAN:
        break
    name = row["task_name"]
    if name not in buckets and sum(len(v) >= PER_TASK for v in buckets.values()) >= N_TASKS:
        continue  # sudah cukup task penuh, abaikan task baru
    if len(buckets[name]) < PER_TASK:
        buckets[name].append(row)
    if sum(len(v) >= PER_TASK for v in buckets.values()) >= N_TASKS:
        break

full_tasks = [name for name, v in buckets.items() if len(v) >= PER_TASK][:N_TASKS]
selected = [r for name in full_tasks for r in buckets[name][:PER_TASK]]
print(f"Task terkumpul: {len(full_tasks)} | total instance eval: {len(selected)}")

# Template sama dengan saat fine-tuning (PROMPT_TEMPLATES_WITH_INPUT varian Alpaca),
# supaya model fine-tuned melihat format yang dikenalnya -> perbandingan adil.
SUPERNI_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n"
)

scorer_ni = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
ni_base, ni_ft = [], []

for row in tqdm(selected, desc="SuperNI base vs ft"):
    ref = (row["targets"] or "").strip()
    if not ref:
        continue
    prompt = SUPERNI_TEMPLATE.format(instruction=row["definition"], input=row["inputs"])
    out_base = generate_text(prompt, use_adapter=False)
    out_ft = generate_text(prompt, use_adapter=True)
    ni_base.append(scorer_ni.score(ref, out_base)["rougeL"].fmeasure)
    ni_ft.append(scorer_ni.score(ref, out_ft)["rougeL"].fmeasure)

print(f"\nSuperNI subset — {len(ni_ft)} instance dari {len(full_tasks)} task")
print(f"ROUGE-L base (tanpa fine-tune) : {np.mean(ni_base)*100:.1f}")
print(f"ROUGE-L fine-tuned            : {np.mean(ni_ft)*100:.1f}")
print(f"Selisih (fine-tuned - base)   : {(np.mean(ni_ft) - np.mean(ni_base))*100:+.1f}")
print("\nBandingkan paper Table 3: GPT-3 vanilla 6.8 -> +SELF-INSTRUCT 39.9")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/757 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/119 [00:00<?, ?it/s]

Task terkumpul: 10 | total instance eval: 100


SuperNI base vs ft:   0%|          | 0/100 [00:00<?, ?it/s]


SuperNI subset — 100 instance dari 10 task
ROUGE-L base (tanpa fine-tune) : 33.8
ROUGE-L fine-tuned            : 45.4
Selisih (fine-tuned - base)   : +11.6

Bandingkan paper Table 3: GPT-3 vanilla 6.8 -> +SELF-INSTRUCT 39.9


### Inference (Demo Model)

Demo kualitatif memakai contoh dari data latih dengan sampling acak (`temperature=1.5`).

In [12]:
FastLanguageModel.for_inference(model)

from transformers import TextStreamer

import json
with open("finetuning_data.jsonl") as f:
    finetuning_data = [json.loads(line) for line in f]

sample = finetuning_data[0]
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
inputs = text_tokenizer(sample['prompt'], return_tensors="pt").to("cuda")

print("Prompt:")
print(sample['prompt'])
print("\nOutput model:")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=1.5,
    min_p=0.1,
)

print("\nExpected completion:")
print(sample['completion'].strip())

Prompt:
Rewrite the sentence "The cat sat on the mat" to sound more poetic.

Output:


Output model:
 The feline perched upon the woven tapestry.<|endoftext|>

Expected completion:
Upon the woven mat, the cat reclined in quiet repose.


### Simpan Model

Simpan LoRA adapters saja. Untuk deployment, merge ke 16bit atau konversi ke GGUF untuk dipakai di llama.cpp/Ollama.

In [13]:
model.save_pretrained("qwen_selfinstruct_lora")
tokenizer.save_pretrained("qwen_selfinstruct_lora")
# model.push_to_hub("YOUR_USERNAME/qwen_selfinstruct_lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("YOUR_USERNAME/qwen_selfinstruct_lora", token="YOUR_HF_TOKEN")

# Merge ke 16bit untuk vLLM/inference server
if False: model.save_pretrained_merged("qwen_selfinstruct_16bit", tokenizer)
if False: model.push_to_hub_merged("YOUR_USERNAME/qwen_selfinstruct_16bit", tokenizer, token="YOUR_HF_TOKEN")

Unsloth: Restored added_tokens_decoder metadata in qwen_selfinstruct_lora/tokenizer_config.json.


### Kemas Output untuk Diunduh

Cell terakhir membundel artefak (data hasil generasi, `finetuning_data.jsonl`, sampel review manual, dan folder checkpoint `outputs/`) jadi satu `selfinstruct_outputs.zip`.

In [15]:
import os, zipfile

artifacts = [
    "selfinstruct_lora_vast_v3.ipynb",   # notebook ini sendiri
    "finetuning_data.jsonl",
    "generated_data.jsonl",
    "manual_review_sample.jsonl",
    "outputs",  # folder checkpoint LoRA hasil training
]

zip_path = "selfinstruct_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in artifacts:
        if os.path.isfile(item):
            zf.write(item)
        elif os.path.isdir(item):
            for root, _, files in os.walk(item):
                for name in files:
                    zf.write(os.path.join(root, name))
        else:
            print("Lewati (tidak ada):", item)

print("Zip siap:", os.path.abspath(zip_path))
print("Ukuran:", round(os.path.getsize(zip_path) / 1024 / 1024, 2), "MB")

Zip siap: /workspace/selfinstruct_outputs.zip
Ukuran: 453.98 MB
